# 6. Sliding Window Maximum
**Difficulty:** 🔴 Hard · **Topic:** Arrays / Queues · **Pattern:** Monotonic (decreasing) deque (LeetCode 239)

> **DevRev context:** you're tracking a rolling metric over time — concurrent open tickets, request latency, queue depth — and you want the **peak value in every sliding window** of the last `k` samples (a rolling max for a live dashboard or an SLA-breach alarm). Recomputing the max for each window is quadratic; a **monotonic deque** streams the answer in one pass.

## 💡 Concepts

**Core concept(s):** A **monotonic deque** — a double-ended queue that holds *candidate* indices in **decreasing value order**, so the current window's maximum is always at the front.

**Why it applies here:** When a new element arrives, any earlier element **smaller** than it can never be the max again (the newcomer is bigger *and* stays in the window longer) — so we drop them from the back. And when the window slides past an index, we drop it from the front. What's left at the front is always the window max.

**Key intuition:** *"A smaller, older element is dominated by a bigger, newer one — evict it."* Each index is pushed and popped at most once → linear.

---

### 📚 What is a deque?
A **deque** ("deck") is a double-ended queue: **O(1)** push/pop at **both** ends. Python's `collections.deque` gives `append`/`pop` (back) and `appendleft`/`popleft` (front). We need both ends: the back to evict dominated candidates, the front to read the max and drop expired indices.

### 📚 What is a *monotonic* deque?
A deque we deliberately keep **sorted** (here, values decreasing from front to back) by evicting anything that would break the order *before* we push. That invariant is what makes the max a simple front-read instead of a scan.

---

**Prerequisite knowledge:**
- Deque push/pop at both ends in O(1).
- Storing **indices** (not values) so we can tell when an element falls out of the window.
- The "dominated element" idea: bigger-and-newer beats smaller-and-older.

## 📝 Problem

Given an array `nums` and a window size `k`, return a list of the **maximum of each contiguous window** of size `k` as the window slides left to right.

**Example**
```
nums = [1, 3, -1, -3, 5, 3, 6, 7], k = 3
windows:
 [1  3 -1]-3  5  3  6  7   -> 3
  1 [3 -1 -3] 5  3  6  7   -> 3
  1  3[-1 -3  5]3  6  7    -> 5
  1  3 -1[-3  5  3]6  7    -> 5
  1  3 -1 -3 [5  3  6]7    -> 6
  1  3 -1 -3  5 [3  6  7]  -> 7
-> [3, 3, 5, 5, 6, 7]
```

> Two approaches: a naive **max-per-window** `O(n*k)` and a **monotonic deque** `O(n)`.

### Approach 1 — Recompute Max Per Window (worst)

For each window position, scan its `k` elements and take the max. Simple and obviously correct, but it re-scans overlapping elements every step → `O(n*k)` (quadratic when `k ~ n/2`).

In [ ]:
from typing import List

def max_sliding_window_naive(nums: List[int], k: int) -> List[int]:
    n = len(nums)
    if n == 0 or k == 0:
        return []
    out = []
    for start in range(n - k + 1):         # each window start
        out.append(max(nums[start:start + k]))   # rescan k elements -> O(k) each
    return out                             # total O(n*k)

### Approach 2 — Monotonic Deque (optimal)

Keep a deque of **indices** whose values are strictly decreasing. Before pushing a new index, pop smaller values off the **back** (they're dominated). Pop the **front** when it slides out of the window. The front index always holds the current window's max. Each index enters and leaves the deque once → `O(n)`.

In [ ]:
from typing import List
from collections import deque

def max_sliding_window_fast(nums: List[int], k: int) -> List[int]:
    n = len(nums)
    if n == 0 or k == 0:
        return []
    dq = deque()          # holds INDICES, values nums[dq] strictly decreasing front->back
    out = []
    for i, x in enumerate(nums):
        # 1) Evict smaller-or-equal values from the back: a newer, bigger element
        #    dominates them (bigger AND stays in the window longer).
        while dq and nums[dq[-1]] <= x:
            dq.pop()
        dq.append(i)                       # x is a fresh max candidate
        # 2) Drop the front if it has slid out of the window [i-k+1 .. i].
        if dq[0] <= i - k:
            dq.popleft()
        # 3) Once the first full window is formed, the front is its maximum.
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out

In [ ]:
# Correctness check
cases = [
    ([1, 3, -1, -3, 5, 3, 6, 7], 3, [3, 3, 5, 5, 6, 7]),
    ([1], 1, [1]),
    ([9, 8, 7, 6], 2, [9, 8, 7]),          # strictly decreasing
    ([1, 2, 3, 4], 2, [2, 3, 4]),          # strictly increasing
    ([4, 4, 4, 4], 2, [4, 4, 4]),          # all equal
    ([], 3, []),
]
for nums, k, expected in cases:
    a = max_sliding_window_naive(nums, k)
    b = max_sliding_window_fast(nums, k)
    assert a == b == expected, f"mismatch on {nums}, k={k}: naive={a} fast={b} exp={expected}"
print("All tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` (with `k = n // 2`) and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)`         | ≈ **2×** |
| `O(n log n)`   | ≈ **2×** (slightly more) |
| `O(n*k) ~ O(n^2)` | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # Strictly decreasing values with k = n//2:
    #  - naive rescans ~k elements per window  -> O(n*k) ~ O(n^2/2)
    #  - the deque never pops from the back (each new value is smaller), so it
    #    still does O(n) total work -> the gap is the whole point.
    nums = list(range(n, 0, -1))
    return (nums, n // 2)

solutions = {
    "per-window O(n*k)": max_sliding_window_naive,
    "deque      O(n)  ": max_sliding_window_fast,
}
sizes = [200, 400, 800, 1600]

benchmark(solutions, make_worst_case, sizes, plot=True)

## 🧩 Patterns Learned

- **Monotonic deque = O(1) rolling extreme:** keep candidates in decreasing order so the max is always the front; each index is pushed/popped once → O(n) overall.
- **Store indices, not values:** you need the index to know when a candidate has slid out of the window (`front <= i - k`).
- **"Dominated" pruning:** a smaller *older* element can never beat a bigger *newer* one — evict it from the back immediately.
- **Amortized O(n):** the inner `while` looks quadratic but every element leaves the deque at most once, so the total work is linear.
- **Signal:** "max/min over every sliding window", "rolling peak", "nearest greater/smaller element" — all monotonic-stack/deque territory.
- **DevRev / related:** rolling peak load / max latency for a live dashboard or SLA alarm; Sliding Window Minimum (flip the comparison); Largest Rectangle in Histogram; Daily Temperatures (monotonic stack).
- **Common pitfalls:** (1) storing values instead of indices (can't expire the window); (2) using `<` vs `<=` inconsistently when evicting equals; (3) reading the max before the first full window has formed (`i >= k - 1` guard).